In [3]:
import feedparser
import pandas as pd
import tmdbsimple as tmdb
from sklearn.metrics.pairwise import cosine_similarity
import time
import os
from dotenv import load_dotenv

tmdb.API_KEY = os.getenv("TMDB_API_KEY")
tmdb.REQUESTS_TIMEOUT = 5 

In [4]:
df = pd.read_csv(r"E:\Python\movie-reccomender\ratings.csv")
 
# Parse date and clean up
df["entry_published"] = pd.to_datetime(df["Date"]).dt.strftime("%a, %-d %b %Y %H:%M:%S +0000")
df = df.rename(columns={"Name": "entry_title", "Rating": "entry_rating"})

In [5]:
def search_tmdb(title, year=None):
    """
    Try a movie search first, then fall back to TV.
    Returns (movie_id, tv_id) — one will always be NaN.
    """
    search = tmdb.Search()
 
    # Movie search
    kwargs = {"query": title}
    if year:
        kwargs["year"] = year
    search.movie(**kwargs)
    if search.results:
        return float(search.results[0]["id"]), float("nan")
 
    # TV search (no year filter — TMDB TV search ignores it anyway)
    search.tv(query=title)
    if search.results:
        return float("nan"), float(search.results[0]["id"])
 
    return float("nan"), float("nan")
 
 
movie_ids, tv_ids = [], []
 
for _, row in df.iterrows():
    title = row["entry_title"]
    # Extract year from the Letterboxd "Year" column if present
    year = row.get("Year")
    year = int(year) if pd.notna(year) and str(year).isdigit() else None
 
    m_id, t_id = search_tmdb(title, year)
    movie_ids.append(m_id)
    tv_ids.append(t_id)
 
    time.sleep(0.25)   # stay well within TMDB rate limits (40 req/10 s)
 
df["movie_id"] = movie_ids
df["tv_id"]    = tv_ids

In [6]:

df = df[["entry_title", "entry_published", "entry_rating", "movie_id", "tv_id"]].copy()
df = df.sort_values("entry_published", ascending=False).reset_index(drop=True)
 

df


# username = "Mouree"
# feed = feedparser.parse(f"https://letterboxd.com/{username}/rss/")
# movieDict = {
#     "entry_title" : [],
#     "entry_published" : [],
#     "entry_rating" : [],
#     "movie_id": [],
#     "tv_id" : [],
# }
# feed.entries[0].tmdb_movieid
# for entry in feed.entries:
    
#     movieDict["entry_title"].append(entry.letterboxd_filmtitle)
#     movieDict["entry_published"].append(entry.published)
#     try:
#         movieDict["movie_id"].append(entry.tmdb_movieid)
#         movieDict["tv_id"].append(pd.NA)
#     except:
#         movieDict["movie_id"].append(pd.NA)
#         movieDict["tv_id"].append(entry.tmdb_tvid)
#     try:
#         movieDict["entry_rating"].append(entry.letterboxd_memberrating)

#     except:
#         movieDict["entry_rating"].append(0)


# df = pd.DataFrame.from_dict(movieDict, orient='columns')

# df['movie_id'] = pd.to_numeric(df['movie_id'])
# df['tv_id'] = pd.to_numeric(df['tv_id']) 


# df

#todo, filter duplicates

,entry_title,entry_published,entry_rating,movie_id,tv_id
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN
...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN


In [7]:
movie = tmdb.TV(13916)
response = movie.info()
# test = movie.genres
# for genre in test:
#     print(genre["name"])

In [8]:

movie_df = df.dropna(subset = ["movie_id"])
movie_df['tv_id'] = pd.to_numeric(movie_df['tv_id'])
movie_df['movie_id'] = pd.to_numeric(movie_df['movie_id'])



movie_df['genres'] = None  # resets the column to object dtype

for movieId in movie_df['movie_id']:
    movie = tmdb.Movies(int(movieId))
    response = movie.info()
    idx = movie_df[movie_df['movie_id'] == movieId].index[0]
    movie_df.at[idx, 'genres'] = ', '.join([g['name'] for g in movie.genres])
    time.sleep(0.1)

movie_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN,Documentary
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN,"Science Fiction, Adventure"
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN,"Romance, Comedy, Drama"
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN,"Action, Adventure, Comedy, Fantasy"
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN,"Crime, Horror, Mystery"
...,...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN,"Action, Crime, Thriller"
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN,"Crime, Drama, Thriller"
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN,"Romance, Comedy, Drama"
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN,"Crime, Drama, Comedy"


In [9]:
tv_df = df.dropna(subset = ["tv_id"])
tv_df['movie_id'] = pd.to_numeric(tv_df['movie_id'])
tv_df['tv_id'] = pd.to_numeric(tv_df['tv_id'])

tv_df['genres'] = None  # resets the column to object dtype

for tvId in tv_df['tv_id']:
    tv = tmdb.TV(int(tvId))
    response = tv.info()
    idx = tv_df[tv_df['tv_id'] == tvId].index[0]
    tv_df.at[idx, 'genres'] = ', '.join([g['name'] for g in tv.genres])
    time.sleep(0.1)
tv_df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres
13,Frieren: Beyond Journey's End,2026-01-19 00:00:00,5.0,NaN,209867.0,"Animation, Action & Adventure, Drama, Sci-Fi &..."
60,Chainsaw Man,2025-05-19 00:00:00,4.0,NaN,114410.0,"Animation, Action & Adventure, Sci-Fi & Fantas..."


In [10]:
movie_df["is_movie"] = True
tv_df["is_movie"] = False
df["genres"] = None

df = df.merge(movie_df[['movie_id', 'genres', 'is_movie']], on='movie_id', how='left')
df = df.merge(tv_df[['tv_id', 'genres', "is_movie"]], on='tv_id', how='left')

df['genres'] = df['genres_x'].fillna(df['genres_y'])
df['is_movie'] = df['is_movie_x'].fillna(df['is_movie_y'])
df = df.drop(columns=['genres_x', 'genres_y', 'is_movie_x', 'is_movie_y'])



In [11]:
encoded_df = df['genres'].str.get_dummies(sep=', ')
df = pd.concat([df, encoded_df], axis=1)
df

,entry_title,entry_published,entry_rating,movie_id,tv_id,genres,is_movie,Action,Adventure,Animation,...,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western
0,Checkpoint Zoo,2026-04-15 00:00:00,4.0,1176733.0,NaN,Documentary,True,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Project Hail Mary,2026-04-13 00:00:00,4.0,687163.0,NaN,"Science Fiction, Adventure",True,0,1,0,...,0,0,0,0,0,0,1,0,0,0
2,The Drama,2026-04-06 00:00:00,4.0,1325734.0,NaN,"Romance, Comedy, Drama",True,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,Big Trouble in Little China,2026-02-14 00:00:00,3.0,6978.0,NaN,"Action, Adventure, Comedy, Fantasy",True,1,1,0,...,1,0,0,0,0,0,0,0,0,0
4,Scream,2026-02-14 00:00:00,4.0,4232.0,NaN,"Crime, Horror, Mystery",True,0,0,0,...,0,0,1,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,The Dark Knight,2024-02-15 00:00:00,5.0,155.0,NaN,"Action, Crime, Thriller",True,1,0,0,...,0,0,0,0,0,0,0,1,0,0
119,Nightcrawler,2024-02-07 00:00:00,4.5,242582.0,NaN,"Crime, Drama, Thriller",True,0,0,0,...,0,0,0,0,0,0,0,1,0,0
120,Risky Business,2024-01-22 00:00:00,4.0,9346.0,NaN,"Romance, Comedy, Drama",True,0,0,0,...,0,0,0,0,0,1,0,0,0,0
121,The Wolf of Wall Street,2024-01-16 00:00:00,4.0,106646.0,NaN,"Crime, Drama, Comedy",True,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# genre_df = encoded_df.copy()

# rating_df = df[["entry_title", "entry_rating"]].copy()
# rating_df["entry_rating"] = pd.to_numeric(rating_df["entry_rating"])

# weights = genre_df.T.dot(rating_df["entry_rating"])  # compute weights with integer index

# genre_df.index = df['movie_id'].values  # set movie_id as index after weights are computed
# weights

genre_df = encoded_df.copy()

rating_df = df[["entry_title", "entry_rating"]].copy()
rating_df["entry_rating"] = pd.to_numeric(rating_df["entry_rating"])

weights = genre_df.T.dot(rating_df["entry_rating"])  # compute weights with integer index

# Only assign movie_id where it exists, drop TV rows
genre_df.index = df['movie_id'].values
genre_df = genre_df[genre_df.index.notna()]  # drop NaN (TV show) rows
genre_df.index = genre_df.index.astype(int)
weights

Action             134.0
Adventure           78.5
Animation           66.0
Comedy             112.5
Crime              141.0
Documentary          9.0
Drama              253.0
Family               5.5
Fantasy             52.0
History              4.0
Horror              69.0
Music                7.0
Mystery             58.5
Romance             51.5
Science Fiction    121.0
Thriller           175.5
War                  8.0
Western             17.5
dtype: float64

In [13]:
movie = tmdb.Movies()
movies = []
page = 1
seen_ids = set(df['movie_id'].dropna().astype(int).tolist())

while len(movies) < 100:
    response = movie.top_rated(page=page)
    for item in response['results']:
        if item['id'] not in seen_ids:
            movies.append(item)
    page += 1

print(response["results"][0]["title"])

len(movies)

popular_df = pd.DataFrame(movies)
popular_df
    

Saving Private Ryan


,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[12, 16, 10751, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",444.1758,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.939,617
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,50.8475,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30301
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",39.5886,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22871
3,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.0612,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13868
4,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,Schindler's List,en,Schindler's List,The true story of how businessman Oskar Schind...,26.1437,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,False,False,8.568,17406
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,False,/18X5mXS1SpQy5rygEcpbmzUdkVP.jpg,"[18, 10751]",1186532,The Forge,en,The Forge,19 year old Isaiah Wright lives for basketball...,3.8705,/oranxontIQRmhuyQlkoAwxpeBYz.jpg,2024-08-22,False,False,8.201,335
96,False,/miChhr7EXynB2R5JLvMcz2oBgi2.jpg,"[16, 28, 12, 14]",618344,Justice League Dark: Apokolips War,en,Justice League Dark: Apokolips War,Earth is decimated after intergalactic tyrant ...,3.3146,/c01Y4suApJ1Wic2xLmaq1QYcfoZ.jpg,2020-05-05,False,False,8.199,1565
97,False,/wyvUmyzqGOBDyqLHRSukGDjI7bH.jpg,[18],50014,The Help,en,The Help,Aibileen Clark is a middle-aged African-Americ...,12.0159,/3kmfoWWEc9Vtyuaf9v5VipRgdjx.jpg,2011-08-09,False,False,8.199,8992
98,False,/xPpXYnCWfjkt3zzE0dpCNME1pXF.jpg,"[16, 28, 14]",635302,Demon Slayer -Kimetsu no Yaiba- The Movie: Mug...,ja,劇場版「鬼滅の刃」無限列車編,"Tanjiro Kamado, joined with Inosuke Hashibira,...",11.4881,/h8Rb9gBr48ODIwYUttZNYeMWeUU.jpg,2020-10-16,False,False,8.198,4473


In [14]:
genres = tmdb.Genres()
response = genres.movie_list()

merged = {d['id']: d['name'] for d in response['genres']}

popular_df['genres'] = popular_df['genre_ids'].apply(lambda ids: [merged[i] for i in ids if i in merged])
popular_df['genres'] = popular_df['genres'].str.join(', ')

popular_df

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count,genres
0,False,/zMwhWailP1WY7sb6AoE6b8ugoy.jpg,"[12, 16, 10751, 14]",1007757,Swapped,en,Swapped,"A small woodland creature and a majestic bird,...",444.1758,/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg,2026-05-01,False,False,8.939,617,"Adventure, Animation, Family, Fantasy"
1,False,/zfbjgQE1uSd9wiPTX4VzsLi0rGG.jpg,"[18, 80]",278,The Shawshank Redemption,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,50.8475,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,1994-09-23,False,False,8.720,30301,"Drama, Crime"
2,False,/tSPT36ZKlP2WVHJLM4cQPLSzv3b.jpg,"[18, 80]",238,The Godfather,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",39.5886,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,1972-03-14,False,False,8.686,22871,"Drama, Crime"
3,False,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,"[18, 80]",240,The Godfather Part II,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.0612,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,1974-12-20,False,False,8.571,13868,"Drama, Crime"
4,False,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,"[18, 36, 10752]",424,Schindler's List,en,Schindler's List,The true story of how businessman Oskar Schind...,26.1437,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,1993-12-15,False,False,8.568,17406,"Drama, History, War"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,False,/18X5mXS1SpQy5rygEcpbmzUdkVP.jpg,"[18, 10751]",1186532,The Forge,en,The Forge,19 year old Isaiah Wright lives for basketball...,3.8705,/oranxontIQRmhuyQlkoAwxpeBYz.jpg,2024-08-22,False,False,8.201,335,"Drama, Family"
96,False,/miChhr7EXynB2R5JLvMcz2oBgi2.jpg,"[16, 28, 12, 14]",618344,Justice League Dark: Apokolips War,en,Justice League Dark: Apokolips War,Earth is decimated after intergalactic tyrant ...,3.3146,/c01Y4suApJ1Wic2xLmaq1QYcfoZ.jpg,2020-05-05,False,False,8.199,1565,"Animation, Action, Adventure, Fantasy"
97,False,/wyvUmyzqGOBDyqLHRSukGDjI7bH.jpg,[18],50014,The Help,en,The Help,Aibileen Clark is a middle-aged African-Americ...,12.0159,/3kmfoWWEc9Vtyuaf9v5VipRgdjx.jpg,2011-08-09,False,False,8.199,8992,Drama
98,False,/xPpXYnCWfjkt3zzE0dpCNME1pXF.jpg,"[16, 28, 14]",635302,Demon Slayer -Kimetsu no Yaiba- The Movie: Mug...,ja,劇場版「鬼滅の刃」無限列車編,"Tanjiro Kamado, joined with Inosuke Hashibira,...",11.4881,/h8Rb9gBr48ODIwYUttZNYeMWeUU.jpg,2020-10-16,False,False,8.198,4473,"Animation, Action, Fantasy"


In [15]:
popular_df = popular_df.drop(["adult",
    "backdrop_path", 
    "original_language", 
    "original_title", 
    "overview", 
    "poster_path", 
    "softcore",
    "video"], axis= 1)



popular_df

,genre_ids,id,title,popularity,release_date,vote_average,vote_count,genres
0,"[12, 16, 10751, 14]",1007757,Swapped,444.1758,2026-05-01,8.939,617,"Adventure, Animation, Family, Fantasy"
1,"[18, 80]",278,The Shawshank Redemption,50.8475,1994-09-23,8.720,30301,"Drama, Crime"
2,"[18, 80]",238,The Godfather,39.5886,1972-03-14,8.686,22871,"Drama, Crime"
3,"[18, 80]",240,The Godfather Part II,26.0612,1974-12-20,8.571,13868,"Drama, Crime"
4,"[18, 36, 10752]",424,Schindler's List,26.1437,1993-12-15,8.568,17406,"Drama, History, War"
...,...,...,...,...,...,...,...,...
95,"[18, 10751]",1186532,The Forge,3.8705,2024-08-22,8.201,335,"Drama, Family"
96,"[16, 28, 12, 14]",618344,Justice League Dark: Apokolips War,3.3146,2020-05-05,8.199,1565,"Animation, Action, Adventure, Fantasy"
97,[18],50014,The Help,12.0159,2011-08-09,8.199,8992,Drama
98,"[16, 28, 14]",635302,Demon Slayer -Kimetsu no Yaiba- The Movie: Mug...,11.4881,2020-10-16,8.198,4473,"Animation, Action, Fantasy"


In [16]:
popular_df_encoded = popular_df['genres'].str.get_dummies(sep=', ')
popular_df_encoded = pd.concat([popular_df, popular_df_encoded], axis=1)
popular_df_encoded

,genre_ids,id,title,popularity,release_date,vote_average,vote_count,genres,Action,Adventure,...,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western
0,"[12, 16, 10751, 14]",1007757,Swapped,444.1758,2026-05-01,8.939,617,"Adventure, Animation, Family, Fantasy",0,1,...,1,0,0,0,0,0,0,0,0,0
1,"[18, 80]",278,The Shawshank Redemption,50.8475,1994-09-23,8.720,30301,"Drama, Crime",0,0,...,0,0,0,0,0,0,0,0,0,0
2,"[18, 80]",238,The Godfather,39.5886,1972-03-14,8.686,22871,"Drama, Crime",0,0,...,0,0,0,0,0,0,0,0,0,0
3,"[18, 80]",240,The Godfather Part II,26.0612,1974-12-20,8.571,13868,"Drama, Crime",0,0,...,0,0,0,0,0,0,0,0,0,0
4,"[18, 36, 10752]",424,Schindler's List,26.1437,1993-12-15,8.568,17406,"Drama, History, War",0,0,...,0,1,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,"[18, 10751]",1186532,The Forge,3.8705,2024-08-22,8.201,335,"Drama, Family",0,0,...,0,0,0,0,0,0,0,0,0,0
96,"[16, 28, 12, 14]",618344,Justice League Dark: Apokolips War,3.3146,2020-05-05,8.199,1565,"Animation, Action, Adventure, Fantasy",1,1,...,1,0,0,0,0,0,0,0,0,0
97,[18],50014,The Help,12.0159,2011-08-09,8.199,8992,Drama,0,0,...,0,0,0,0,0,0,0,0,0,0
98,"[16, 28, 14]",635302,Demon Slayer -Kimetsu no Yaiba- The Movie: Mug...,11.4881,2020-10-16,8.198,4473,"Animation, Action, Fantasy",1,0,...,1,0,0,0,0,0,0,0,0,0


In [17]:
popular_genre_df = popular_df_encoded.drop(["genre_ids",
    "popularity",
    "title",
    "release_date",
    "vote_average",
    "vote_count",
    "genres"], axis = 1)
popular_genre_df = popular_genre_df.set_index('id')
popular_genre_df

,Action,Adventure,Animation,Comedy,Crime,Drama,Family,Fantasy,History,Horror,Music,Mystery,Romance,Science Fiction,Thriller,War,Western
id,,,,,,,,,,,,,,,,,
1007757,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0
278,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
238,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
240,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0
424,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186532,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0
618344,1,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
50014,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [18]:
print(set(popular_genre_df.columns) - set(weights.index))  # in popular but not in weights
print(set(weights.index) - set(popular_genre_df.columns))  # in weights but not in popular

missing_cols = set(weights.index) - set(popular_genre_df.columns)
for col in missing_cols:
    popular_genre_df[col] = 0

set()
{'Documentary'}


In [28]:
# recommendation_table_df = popular_genre_df.dot(weights) / weights.sum()


# from sklearn.preprocessing import normalize

# genre_df_norm = pd.DataFrame(
#     normalize(genre_df, norm='l1'),  # l1 means each row sums to 1
#     index=genre_df.index,
#     columns=genre_df.columns
# )

# popular_genre_df_norm = pd.DataFrame(
#     normalize(popular_genre_df, norm='l1'),
#     index=popular_genre_df.index,
#     columns=popular_genre_df.columns
# )


similarity_matrix = cosine_similarity(popular_genre_df, genre_df)

rating_weights = (df.dropna(subset='movie_id')
                    .drop_duplicates(subset='movie_id')
                    .assign(movie_id=lambda x: x['movie_id'].astype(int))
                    .set_index('movie_id')['entry_rating']
                    .reindex(genre_df.index)
                    .fillna(0)
                    .astype(float)
                    .values)

baseline = df['entry_rating'].mean()  # or df['entry_rating'].mean()
rating_weights = rating_weights - baseline  # now 2.5-rated films push *away* from similar movies

recommendation_table_df = pd.Series(
    similarity_matrix.dot(rating_weights) / rating_weights.sum(),
    index=popular_genre_df.index
)

# recommendation_table_df
rating_weights

array([-0.13821138, -0.13821138, -0.13821138, -1.13821138, -0.13821138,
       -1.63821138,  0.36178862, -0.13821138,  0.86178862, -1.13821138,
       -0.13821138, -0.63821138,  0.86178862, -1.13821138,  0.86178862,
       -1.13821138, -1.13821138, -0.13821138, -0.63821138,  0.86178862,
        0.36178862, -1.13821138, -0.13821138, -0.13821138,  0.86178862,
        0.86178862,  0.86178862,  0.36178862, -0.63821138,  0.36178862,
        0.36178862,  0.86178862,  0.86178862,  0.36178862, -1.13821138,
       -0.13821138,  0.36178862, -1.13821138,  0.36178862, -0.13821138,
       -1.13821138, -0.63821138,  0.86178862,  0.36178862, -1.13821138,
       -0.13821138,  0.86178862,  0.86178862,  0.86178862, -0.13821138,
       -0.13821138, -0.13821138, -0.13821138,  0.86178862,  0.86178862,
        0.86178862, -0.13821138, -0.13821138,  0.36178862,  0.86178862,
       -0.13821138, -2.13821138, -0.13821138, -0.13821138, -0.13821138,
       -0.63821138, -0.13821138,  0.86178862, -0.13821138, -0.13

In [29]:
recommendation_table_df.sort_values(ascending=False, inplace=True)

recommendation_table_df.head(20)

id
24188      8.923725
105        8.400928
121        7.973486
120        7.973486
1181678    6.537723
42269      6.531970
644479     6.531970
77338      6.531970
637        6.531970
1356039    6.310875
490132     6.070986
1891       6.056297
299534     6.056297
27205      6.056297
11         6.056297
299536     6.056297
654299     5.847627
40096      5.333828
13         5.051305
19404      5.051305
dtype: float64

In [30]:
top_recommendations = recommendation_table_df.head(20).index

for movie_id in top_recommendations:
    idx = popular_genre_df.index.get_loc(movie_id)
    most_similar_idx = similarity_matrix[idx].argmax()
    most_similar_id = genre_df.index[most_similar_idx]
    title = df[df['movie_id'] == most_similar_id]['entry_title'].values
    print(f"{movie_id} most similar to: {title}")

24188 most similar to: <StringArray>
['Anaconda']
Length: 1, dtype: str
105 most similar to: <StringArray>
['The Drama']
Length: 1, dtype: str
121 most similar to: <StringArray>
['Speed Racer']
Length: 1, dtype: str
120 most similar to: <StringArray>
['Speed Racer']
Length: 1, dtype: str
1181678 most similar to: <StringArray>
['Glass Onion']
Length: 1, dtype: str
42269 most similar to: <StringArray>
['Checkpoint Zoo']
Length: 1, dtype: str
644479 most similar to: <StringArray>
['Checkpoint Zoo']
Length: 1, dtype: str
77338 most similar to: <StringArray>
['Checkpoint Zoo']
Length: 1, dtype: str
637 most similar to: <StringArray>
['Checkpoint Zoo']
Length: 1, dtype: str
1356039 most similar to: <StringArray>
['The Fantastic 4: First Steps']
Length: 1, dtype: str
490132 most similar to: <StringArray>
['Being John Malkovich']
Length: 1, dtype: str
1891 most similar to: <StringArray>
['The Lord of the Rings: The Return of the King']
Length: 1, dtype: str
299534 most similar to: <StringArray

In [31]:
copy = popular_df.copy(deep=True)

# Then we set its index to movieId
copy = copy.set_index('id', drop=True)

# Next we enlist the top 20 recommended movieIds we defined above
top_20_index = recommendation_table_df.index[:20].tolist()

# finally we slice these indices from the copied movies df and save in a variable
recommended_movies = copy.loc[top_20_index, :]

# Now we can display the top 20 movies in descending order of preference
recommended_movies

,genre_ids,title,popularity,release_date,vote_average,vote_count,genres
id,,,,,,,
24188,"[18, 35, 12]",Il Sorpasso,1.7815,1962-12-05,8.200,835,"Drama, Comedy, Adventure"
105,"[12, 35, 878]",Back to the Future,19.3237,1985-07-03,8.326,21721,"Adventure, Comedy, Science Fiction"
121,"[12, 14, 28]",The Lord of the Rings: The Two Towers,23.1125,2002-12-18,8.417,23801,"Adventure, Fantasy, Action"
120,"[12, 14, 28]",The Lord of the Rings: The Fellowship of the Ring,30.5824,2001-12-18,8.433,27454,"Adventure, Fantasy, Action"
1181678,"[35, 10749]",¿Quieres ser mi hijo?,11.3329,2023-09-21,8.453,330,"Comedy, Romance"
42269,"[18, 35]",We All Loved Each Other So Much,3.8900,1974-12-21,8.300,644,"Drama, Comedy"
644479,"[18, 35]",Dedicated to my ex,3.8696,2019-11-01,8.300,513,"Drama, Comedy"
77338,"[18, 35]",The Intouchables,16.0731,2011-11-02,8.269,18371,"Drama, Comedy"
637,"[35, 18]",Life Is Beautiful,13.1090,1997-12-20,8.438,13950,"Comedy, Drama"
